## Example 1: Spatial weights creation

In [ ]:
import libpysal
from libpysal import weights, examples
import numpy as np

# Example 1: Spatial weights creation
print("Testing PySAL spatial weights:")

# Get example points
points = [(0, 0), (1, 0), (1, 1), (0, 1), (2, 1)]

# Create KNN weights
w_knn = weights.KNN.from_array(np.array(points), k=2)
print(f"✓ Created KNN weights: {w_knn.n} observations, k={2}")
print(f"✓ Neighbors of point 0: {w_knn.neighbors[0]}")
print(f"✓ Weights of point 0: {w_knn.weights[0]}")

# Create distance band weights
w_dist = weights.DistanceBand.from_array(np.array(points), threshold=1.5)
print(f"✓ Created distance band weights: threshold=1.5")
print(f"✓ Neighbors of point 0: {w_dist.neighbors[0]}")
print(f"✓ Cardinalities: {w_dist.cardinalities}")

# Get weight properties
print(f"✓ Is symmetric: {w_knn.asymmetries == ()}")
print(f"✓ Sparsity: {w_knn.pct_nonzero:.2f}%")
assert w_knn.n == 5 and all(len(v) == 2 for v in w_knn.neighbors.values())
assert sorted(w_knn.neighbors[0]) == [1, 3]   # the two points at distance 1 from (0, 0)
assert sorted(w_dist.neighbors[0]) == [1, 2, 3]
assert w_dist.cardinalities == {0: 3, 1: 4, 2: 4, 3: 3, 4: 2}
assert w_knn.pct_nonzero == 40.0              # 10 of the 25 possible links

## Example 2: Weight transformations and operations

In [ ]:
from libpysal import weights
import numpy as np

# Example 2: Weight transformations and operations
print("Testing PySAL weight transformations:")

# Create weights
points = np.array([(0, 0), (1, 0), (1, 1), (0, 1)])
w = weights.KNN.from_array(points, k=2)

print(f"✓ Original weights sum: {sum([sum(wts) for wts in w.weights.values()]):.1f}")
print(f"✓ Transform: {w.transform}")

# Row-standardize weights
w.transform = 'r'
print(f"✓ Row-standardized weights sum: {sum([sum(wts) for wts in w.weights.values()]):.1f}")
print(f"✓ Weights of point 0: {w.weights[0]}")

# Binary transformation
w.transform = 'b'
print(f"✓ Binary weights: {w.weights[0]}")

# Spatial lag
y = np.array([10, 20, 30, 40])
w.transform = 'r'
lag = weights.lag_spatial(w, y)
print(f"✓ Spatial lag of y: {lag}")
print(f"✓ Original y: {y}")
# Each corner of the unit square has its two edge neighbours, so each row-standardised weight is 0.5
assert sorted(w.neighbors[0]) == [1, 3]
assert w.weights[0] == [0.5, 0.5]
assert sum(sum(wts) for wts in w.weights.values()) == 4.0   # every row sums to 1
assert np.allclose(lag, [30, 20, 30, 20])                   # mean of each point's two neighbours

## Example 3: Spatial autocorrelation with Moran's I

In [ ]:
from libpysal import weights
from esda import Moran
import numpy as np

# Example 3: Spatial autocorrelation with Moran's I
print("Testing PySAL Moran's I autocorrelation:")

# Create spatial data
np.random.seed(12345)
points = np.array([(i, j) for i in range(4) for j in range(4)])
w = weights.KNN.from_array(points, k=3)
w.transform = 'r'

# Generate spatially autocorrelated data
y = np.random.normal(50, 10, 16)

# Calculate Moran's I
moran = Moran(y, w)
print(f"✓ Moran's I: {moran.I:.4f}")
print(f"✓ Expected I: {moran.EI:.4f}")
print(f"✓ Variance: {moran.VI_norm:.6f}")
print(f"✓ Z-score: {moran.z_norm:.4f}")
print(f"✓ P-value: {moran.p_norm:.4f}")
print(f"✓ Interpretation: {'Significant' if moran.p_norm < 0.05 else 'Not significant'} spatial autocorrelation")
# Under no autocorrelation E[I] = -1/(n-1); the z-score is (I - E[I]) / sqrt(Var[I])
assert np.isclose(moran.EI, -1 / 15)
assert np.isclose(moran.z_norm, (moran.I - moran.EI) / np.sqrt(moran.VI_norm))
assert 0 < moran.p_norm < 1
assert np.isclose(moran.I, 0.1084, atol=1e-4)   # fixed seed above

## Example 4: Local spatial autocorrelation (LISA)

In [ ]:
from libpysal import weights
from esda import Moran, Moran_Local
import numpy as np

# Example 4: Local spatial autocorrelation (LISA)
print("Testing PySAL Local Moran's I (LISA):")

# Create spatial data
np.random.seed(12345)
points = np.array([(i, j) for i in range(5) for j in range(5)])
w = weights.KNN.from_array(points, k=4)
w.transform = 'r'

# Generate data with clusters
y = np.random.normal(50, 10, 25)
y[0:5] = y[0:5] + 20  # High cluster
y[20:25] = y[20:25] - 20  # Low cluster

# Calculate Local Moran's I
lisa = Moran_Local(y, w)
print(f"✓ Local Moran's I values: {lisa.Is[:5]}")
print(f"✓ P-values: {lisa.p_sim[:5]}")
print(f"✓ Quadrants: {lisa.q[:5]}")  # 1=HH, 2=LH, 3=LL, 4=HL

# Count significant clusters
sig = lisa.p_sim < 0.05
print(f"✓ Significant locations: {np.sum(sig)}/{len(y)}")
print(f"✓ Cluster types: HH={np.sum((lisa.q==1) & sig)}, LL={np.sum((lisa.q==3) & sig)}")
# With row-standardised weights the local statistics sum to (n - 1) times the global Moran's I
assert np.isclose(lisa.Is.sum() / (len(y) - 1), Moran(y, w).I)
assert (lisa.q[:5] == 1).all()   # the raised first row forms a high-high cluster
assert set(lisa.q) <= {1, 2, 3, 4}
assert ((lisa.p_sim > 0) & (lisa.p_sim <= 1)).all()

## Example 5: Geary's C and join counts

In [ ]:
from libpysal import weights
from esda import Geary, Gamma, Join_Counts
import numpy as np

# Example 5: Geary's C and join counts
print("Testing PySAL additional spatial statistics:")

# Setup
np.random.seed(12345)
points = np.array([(i, j) for i in range(4) for j in range(4)])
w = weights.KNN.from_array(points, k=3)
w.transform = 'r'

# Geary's C statistic
y = np.random.normal(50, 10, 16)
geary = Geary(y, w)
print(f"✓ Geary's C: {geary.C:.4f}")
print(f"✓ Expected C: {geary.EC:.4f}")
print(f"✓ P-value: {geary.p_norm:.4f}")

# Join counts for binary data
y_binary = (y > y.mean()).astype(int)
jc = Join_Counts(y_binary, w)
print(f"✓ Join Counts BB: {jc.bb}")
print(f"✓ Join Counts WW: {jc.ww}")
print(f"✓ Join Counts BW: {jc.bw}")
assert geary.EC == 1.0           # Geary's C has expectation 1 under no autocorrelation
assert 0 < geary.p_norm < 1
# Every join (pair of neighbours) is black-black, white-white or black-white
assert jc.bb + jc.ww + jc.bw == jc.J == 24.0